In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, SimpleRNN, Dense, Conv1D, Flatten, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from itertools import combinations
import warnings
warnings.filterwarnings("ignore")

# ================================================
# BASELINE INFLATION FORECASTING WITHOUT FILTERS
# Scientific approach using raw Fed leading indicators
# ================================================

print("🚀 BASELINE INFLATION FORECASTING WITHOUT FILTERS")
print("="*75)
print("✅ NO inflation as predictor (avoiding data leakage)")
print("✅ Raw Fed indicators without preprocessing filters")  
print("✅ Multiple evaluation metrics (MSE, MAE, R², MAPE)")
print("✅ Baseline for comparison with filtered approaches")
print("✅ Direct neural network learning from unfiltered data")
print("="*75)

# ================================================
# 1. SCIENTIFIC SETUP - LEADING FED INDICATORS
# ================================================

def load_and_document_data(file_path):
    """
    Load inflation forecasting data with Fed leading indicators
    
    Leading Indicators (RAW - NO FILTERING):
    - USM2: Money Supply M2 (raw monetary policy data)
    - USPPIYY: Producer Price Index YoY (raw inflation pressure data)  
    - INDPRO: Industrial Production Index (raw economic activity data)
    - UNRATE: Unemployment Rate (raw labor market data)
    - USOIL: Oil Prices (raw commodity price data)
    
    Target:
    - Annual Inflation Rate (target for baseline forecasting)
    
    Note: No preprocessing filters applied - neural networks learn directly
    from raw economic data with inherent noise and volatility
    """
    print("\n📊 LOADING RAW FED INDICATORS (NO FILTERING)")
    print("-" * 50)
    
    try:
        data = pd.read_csv(file_path)
        print(f"✅ Data loaded: {data.shape[0]} observations, {data.shape[1]} features")
    except FileNotFoundError:
        print(f"❌ File not found: {file_path}")
        print("📁 Please check the file path and ensure the CSV file exists")
        return None
    except Exception as e:
        print(f"❌ Error loading data: {e}")
        return None
    
    # Handle missing values
    missing_before = data.isnull().sum().sum()
    print(f"📋 Missing values before: {missing_before}")
    
    if missing_before > 0:
        data = data.interpolate(method='linear')
        data = data.fillna(method='ffill').fillna(method='bfill')
        
    missing_after = data.isnull().sum().sum()
    print(f"✅ Missing values after interpolation: {missing_after}")
    
    return data

# ================================================
# 2. SCIENTIFIC VARIABLE DEFINITION - SAME AS FILTERED CODES
# ================================================

# LEADING FED INDICATORS (no inflation as predictor)
LEADING_INDICATORS = [
    'USM2',      # Money Supply M2 - Monetary policy stance
    'USPPIYY',   # Producer Price Index YoY - Cost pressures  
    'INDPRO',    # Industrial Production - Economic activity
    'UNRATE',    # Unemployment Rate - Labor market conditions
    'USOIL'      # Oil Prices - Supply shocks and commodity inflation
]

TARGET_VARIABLE = 'Annual Inflation Rate'

print(f"\n🎯 BASELINE RESEARCH DESIGN (NO FILTERS):")
print(f"Leading Fed Indicators (Raw): {LEADING_INDICATORS}")
print(f"Target Variable: {TARGET_VARIABLE}")
print(f"\n💡 Research Question:")
print(f"How do neural networks perform on raw Fed indicators without preprocessing?")
print(f"This provides the baseline for comparison with filtered approaches.")

# ================================================
# 3. MODEL CONFIGURATIONS
# ================================================

MODEL_CONFIGS = {
    'LSTM': {
        'units': 50,
        'activation': 'tanh',
        'dropout_rate': 0.2,
        'epochs': 100,
        'batch_size': 32,
        'layers': 2,
        'architecture': 'Sequential: LSTM(50) → LSTM(50) → Dense(1)',
        'optimal_for': 'Long-term dependencies in raw economic data'
    },
    'GRU': {
        'units': 50,
        'activation': 'tanh', 
        'dropout_rate': 0.2,
        'epochs': 100,
        'batch_size': 32,
        'layers': 2,
        'architecture': 'Sequential: GRU(50) → GRU(50) → Dense(1)',
        'optimal_for': 'Efficient processing of raw temporal patterns'
    },
    'CNN': {
        'filters': 64,
        'kernel_size': 2,
        'activation': 'relu',
        'dense_units': 50,
        'dropout_rate': 0.2,
        'epochs': 100,
        'batch_size': 32,
        'input_type': '1D convolution',
        'architecture': 'Conv1D(64) → Flatten → Dense(50) → Dense(1)',
        'optimal_for': 'Local pattern detection in raw time series'
    },
    'RNN': {
        'units': 50,
        'activation': 'tanh',
        'dropout_rate': 0.2,
        'epochs': 100,
        'batch_size': 32,
        'layers': 2,
        'architecture': 'Sequential: RNN(50) → RNN(50) → Dense(1)',
        'optimal_for': 'Basic sequential processing of raw data'
    },
    'ANN': {
        'units': 50,
        'activation': 'relu',
        'dropout_rate': 0.2,
        'epochs': 100,
        'batch_size': 32,
        'layers': 2,
        'architecture': 'Dense(50) → Dense(50) → Dense(1)',
        'optimal_for': 'Non-linear relationships in raw feature space'
    }
}

print(f"\n🤖 MODEL ARCHITECTURES FOR RAW FED DATA:")
for model_name, config in MODEL_CONFIGS.items():
    print(f"   {model_name}: {config['architecture']}")
    print(f"     └─ Optimal for: {config['optimal_for']}")

# ================================================
# 4. PREPROCESSING FUNCTIONS
# ================================================

def create_sequences(data, seq_length):
    """
    Create sequences for time series prediction from raw Fed data
    
    Input structure (NO FILTERING):
    - Each sequence contains seq_length time steps of raw Fed indicators
    - Features: Raw leading indicators (USM2, USPPIYY, INDPRO, UNRATE, USOIL)
    - Target: Annual Inflation Rate at time t+1
    
    Neural networks must learn to handle noise and extract patterns directly
    """
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i + seq_length, :-1])  # Raw Fed indicators
        y.append(data[i + seq_length, -1])     # Target: inflation at t+1
    return np.array(X), np.array(y)

# ================================================
# 5. MODEL BUILDERS (ENHANCED WITH REGULARIZATION)
# ================================================

def build_lstm_model(input_shape):
    """Build LSTM model for raw Fed economic data"""
    model = Sequential()
    model.add(LSTM(50, return_sequences=True, input_shape=input_shape, activation='tanh'))
    model.add(Dropout(0.2))  # Important for raw noisy data
    model.add(LSTM(50, activation='tanh'))
    model.add(Dropout(0.2))  # Prevent overfitting on noise
    model.add(Dense(1))
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae'])
    return model

def build_gru_model(input_shape):
    """Build GRU model for raw economic time series"""
    model = Sequential()
    model.add(GRU(50, return_sequences=True, input_shape=input_shape, activation='tanh'))
    model.add(Dropout(0.2))
    model.add(GRU(50, activation='tanh'))
    model.add(Dropout(0.2))
    model.add(Dense(1))
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae'])
    return model

def build_cnn_model(input_shape):
    """Build CNN model for 1D convolution on raw Fed sequences"""
    model = Sequential()
    model.add(Conv1D(64, kernel_size=2, activation='relu', input_shape=input_shape))
    model.add(Dropout(0.2))
    model.add(Flatten())
    model.add(Dense(50, activation='relu'))
    model.add(Dropout(0.2))
    model.add(Dense(1))
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae'])
    return model

def build_rnn_model(input_shape):
    """Build RNN model for raw economic time series"""
    model = Sequential()
    model.add(SimpleRNN(50, return_sequences=True, input_shape=input_shape, activation='tanh'))
    model.add(Dropout(0.2))
    model.add(SimpleRNN(50, activation='tanh'))
    model.add(Dropout(0.2))
    model.add(Dense(1))
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae'])
    return model

def build_ann_model(input_shape):
    """Build ANN model for flattened raw Fed sequences"""
    model = Sequential()
    model.add(Dense(50, activation='relu', input_shape=input_shape))
    model.add(Dropout(0.2))
    model.add(Dense(50, activation='relu'))
    model.add(Dropout(0.2))
    model.add(Dense(1))
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae'])
    return model

# ================================================
# 6. COMPREHENSIVE METRICS
# ================================================

def calculate_comprehensive_metrics(y_true, y_pred):
    """Calculate comprehensive evaluation metrics for baseline forecasting"""
    return {
        'mse': mean_squared_error(y_true, y_pred),
        'rmse': np.sqrt(mean_squared_error(y_true, y_pred)),
        'mae': mean_absolute_error(y_true, y_pred),
        'r2': r2_score(y_true, y_pred),
        'mape': mean_absolute_percentage_error(y_true, y_pred) * 100
    }

# ================================================
# 7. ML BASELINE MODELS
# ================================================

def train_ml_baselines(X_train_flat, X_test_flat, y_train, y_test):
    """Train ML baseline models on raw Fed indicators"""
    print("   🌳 Training ML baselines on raw Fed data...")
    ml_results = {}
    
    try:
        # Random Forest
        rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
        rf_model.fit(X_train_flat, y_train)
        rf_pred = rf_model.predict(X_test_flat)
        ml_results['random_forest'] = calculate_comprehensive_metrics(y_test, rf_pred)
        
        # XGBoost
        xgb_model = xgb.XGBRegressor(n_estimators=100, random_state=42, n_jobs=-1)
        xgb_model.fit(X_train_flat, y_train)
        xgb_pred = xgb_model.predict(X_test_flat)
        ml_results['xgboost'] = calculate_comprehensive_metrics(y_test, xgb_pred)
        
        print(f"      ✅ Random Forest: MSE={ml_results['random_forest']['mse']:.6f}")
        print(f"      ✅ XGBoost: MSE={ml_results['xgboost']['mse']:.6f}")
        
    except Exception as e:
        print(f"      ⚠️ ML baseline error: {e}")
        
    return ml_results

# ================================================
# 8. MAIN ANALYSIS - RAW FED INDICATORS
# ================================================

def run_baseline_fed_inflation_forecasting():
    """
    Run comprehensive baseline inflation forecasting with raw Fed indicators
    """
    
    # Load data
    file_path = r"C:\Users\elect\OneDrive\Documentos\Doctorado\Articulo 2 peer review\Data\Inflation_Parameters.csv"
    print(f"📁 Loading Fed economic indicators from: {file_path}")
    
    merged_data = load_and_document_data(file_path)
    
    if merged_data is None:
        print("❌ Cannot proceed without valid Fed data")
        return []
    
    # Verify required columns exist
    required_columns = LEADING_INDICATORS + [TARGET_VARIABLE]
    missing_columns = [col for col in required_columns if col not in merged_data.columns]
    
    if missing_columns:
        print(f"❌ Missing required Fed indicators: {missing_columns}")
        print(f"📋 Available columns: {list(merged_data.columns)}")
        return []
    else:
        print(f"✅ All required Fed indicators found in dataset")
    
    results = []
    seq_length = 10
    
    # Test both normalization methods
    scalers = {
        'MinMax': MinMaxScaler(),
        'Standard': StandardScaler()
    }
    
    print(f"\n🔬 BASELINE FED INFLATION FORECASTING EXPERIMENT")
    print(f"Fed Leading indicators: {LEADING_INDICATORS}")
    print(f"Target: {TARGET_VARIABLE}")
    print(f"Sequence length: {seq_length}")
    print(f"Normalization methods: {list(scalers.keys())}")
    print(f"Filter applied: NONE (raw data)")
    print("-" * 75)
    
    # Calculate total experiments
    total_combinations = sum(1 for r in range(1, len(LEADING_INDICATORS) + 1) 
                           for _ in combinations(LEADING_INDICATORS, r))
    total_experiments = total_combinations * len(scalers)
    current_experiment = 0
    
    # Iterate through all combinations of Fed leading indicators
    for r in range(1, len(LEADING_INDICATORS) + 1):
        for combo in combinations(LEADING_INDICATORS, r):
            for scaler_name, scaler in scalers.items():
                current_experiment += 1
                
                print(f"\n📊 Experiment {current_experiment}/{total_experiments}")
                print(f"    Fed Indicators: {combo}")
                print(f"    Normalization: {scaler_name}")
                
                try:
                    # Prepare dataset with ONLY Fed leading indicators + target
                    selected_features = list(combo) + [TARGET_VARIABLE]
                    selected_data = merged_data[selected_features].copy()
                    
                    print(f"   📊 Selected Fed data shape: {selected_data.shape}")
                    print("   📊 Using RAW data (no filtering applied)")
                    
                    # Normalize raw Fed data (NO FILTERING)
                    scaled_data = scaler.fit_transform(selected_data)
                    
                    # Create sequences for time series modeling
                    X, y = create_sequences(scaled_data, seq_length)
                    
                    if len(X) < 20:  # Minimum samples for reliable training
                        print("   ⚠️ Insufficient data for reliable training")
                        continue
                    
                    # Temporal train-test split (crucial for time series)
                    split_idx = int(len(X) * 0.8)
                    X_train, X_test = X[:split_idx], X[split_idx:]
                    y_train, y_test = y[:split_idx], y[split_idx:]
                    
                    print(f"   📊 Data split: Train={len(X_train)}, Test={len(X_test)}")
                    
                    # Prepare input shapes
                    input_shape_seq = (seq_length, X_train.shape[2])
                    X_train_flat = X_train.reshape(X_train.shape[0], -1)
                    X_test_flat = X_test.reshape(X_test.shape[0], -1)
                    input_shape_ann = (X_train_flat.shape[1],)
                    
                    # Early stopping callback
                    early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
                    
                    # Train all models
                    models_results = []
                    
                    # Neural Network Models
                    neural_models = [
                        ('LSTM', build_lstm_model, X_train, X_test, input_shape_seq),
                        ('GRU', build_gru_model, X_train, X_test, input_shape_seq),
                        ('CNN', build_cnn_model, X_train, X_test, input_shape_seq),
                        ('RNN', build_rnn_model, X_train, X_test, input_shape_seq),
                        ('ANN', build_ann_model, X_train_flat, X_test_flat, input_shape_ann)
                    ]
                    
                    for model_name, model_builder, X_tr, X_te, input_shape in neural_models:
                        print(f"   🤖 Training {model_name} on raw Fed data...")
                        
                        try:
                            model = model_builder(input_shape)
                            history = model.fit(X_tr, y_train, epochs=100, batch_size=32, 
                                              validation_split=0.2, verbose=0, callbacks=[early_stopping])
                            
                            y_pred = model.predict(X_te, verbose=0).flatten()
                            metrics = calculate_comprehensive_metrics(y_test, y_pred)
                            models_results.append((model_name, metrics))
                            
                            print(f"      ✅ {model_name}: MSE={metrics['mse']:.6f}, R²={metrics['r2']:.4f}")
                            
                        except Exception as e:
                            print(f"      ❌ {model_name} failed: {e}")
                    
                    # Train ML baseline models
                    ml_results = train_ml_baselines(X_train_flat, X_test_flat, y_train, y_test)
                    for ml_model, ml_metrics in ml_results.items():
                        models_results.append((ml_model.upper(), ml_metrics))
                    
                    # Store all results
                    for model_name, metrics in models_results:
                        result = {
                            'leading_indicators': combo,
                            'n_indicators': len(combo),
                            'scaler': scaler_name,
                            'model': model_name,
                            'seq_length': seq_length if model_name not in ['RANDOM_FOREST', 'XGBOOST'] else 'N/A',
                            'filter_applied': 'NONE',
                            'data_type': 'RAW',
                            **metrics
                        }
                        results.append(result)
                
                except Exception as e:
                    print(f"   ❌ Experiment failed: {e}")
                    continue
    
    return results

# ================================================
# 9. EXECUTE ANALYSIS AND CREATE VISUALIZATIONS
# ================================================

if __name__ == "__main__":
    print("🚀 STARTING BASELINE FED INFLATION FORECASTING ANALYSIS")
    
    try:
        # Run the analysis
        results = run_baseline_fed_inflation_forecasting()
        
        if not results:
            print("❌ No results generated!")
        else:
            # Convert to DataFrame
            results_df = pd.DataFrame(results)
            
            # Save results
            output_path = r'C:\Users\elect\OneDrive\Documentos\Doctorado\Articulo 2 peer review\results\baseline_raw_fed_inflation_results.csv'
            results_df.to_csv(output_path, index=False)
            print(f"\n💾 Results saved to: {output_path}")
            
            # ================================================
            # 10. BASELINE VISUALIZATION
            # ================================================
            
            print(f"\n📊 CREATING BASELINE FED ANALYSIS VISUALIZATIONS")
            print("-" * 55)
            
            # Create comprehensive visualization
            fig, axes = plt.subplots(2, 3, figsize=(20, 12))
            fig.suptitle('Baseline Inflation Forecasting\n(Raw Fed Leading Indicators - No Filtering)', 
                         fontsize=16, fontweight='bold')
            
            # 1. Model Performance without Filtering (MSE)
            ax1 = axes[0, 0]
            model_mse = results_df.groupby('model')['mse'].mean().sort_values()
            bars1 = ax1.bar(range(len(model_mse)), model_mse.values, 
                            color=plt.cm.Set3(np.linspace(0, 1, len(model_mse))))
            ax1.set_xticks(range(len(model_mse)))
            ax1.set_xticklabels(model_mse.index, rotation=45, ha='right')
            ax1.set_ylabel('Mean Squared Error')
            ax1.set_title('Baseline Raw Data Forecasting')
            ax1.grid(True, alpha=0.3)
            
            # Add value labels
            for bar, value in zip(bars1, model_mse.values):
                ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                        f'{value:.4f}', ha='center', va='bottom', fontsize=9)
            
            # 2. R² without Filtering
            ax2 = axes[0, 1]
            model_r2 = results_df.groupby('model')['r2'].mean().sort_values(ascending=False)
            bars2 = ax2.bar(range(len(model_r2)), model_r2.values,
                            color=plt.cm.Set2(np.linspace(0, 1, len(model_r2))))
            ax2.set_xticks(range(len(model_r2)))
            ax2.set_xticklabels(model_r2.index, rotation=45, ha='right')
            ax2.set_ylabel('R² Score')
            ax2.set_title('Explained Variance (Raw Data)')
            ax2.grid(True, alpha=0.3)
            
            # 3. Fed Indicators Impact (Raw)
            ax3 = axes[0, 2]
            indicator_impact = results_df.groupby('n_indicators')['r2'].agg(['mean', 'std']).reset_index()
            ax3.errorbar(indicator_impact['n_indicators'], indicator_impact['mean'], 
                        yerr=indicator_impact['std'], marker='o', capsize=5, linewidth=2)
            ax3.set_xlabel('Number of Raw Fed Indicators')
            ax3.set_ylabel('R² Score (Mean ± Std)')
            ax3.set_title('Raw Data Complexity vs Performance')
            ax3.grid(True, alpha=0.3)
            
            # 4. Scaler Impact on Raw Fed Data
            ax4 = axes[1, 0]
            scaler_perf = results_df.groupby(['model', 'scaler'])['r2'].mean().unstack()
            scaler_perf.plot(kind='bar', ax=ax4, width=0.8)
            ax4.set_xlabel('Model Type')
            ax4.set_ylabel('R² Score')
            ax4.set_title('Normalization Impact (Raw Data)')
            ax4.legend(title='Scaler')
            ax4.grid(True, alpha=0.3)
            plt.setp(ax4.xaxis.get_majorticklabels(), rotation=45, ha='right')
            
            # 5. Top Raw Fed Configurations
            ax5 = axes[1, 1]
            top_10 = results_df.nsmallest(10, 'mse')
            config_names = [f"{row['model']}\n{'+'.join(row['leading_indicators'][:2])}" 
                           for _, row in top_10.iterrows()]
            bars5 = ax5.barh(range(len(top_10)), top_10['mse'])
            ax5.set_yticks(range(len(top_10)))
            ax5.set_yticklabels(config_names, fontsize=8)
            ax5.set_xlabel('MSE')
            ax5.set_title('Top 10 Raw Data Configurations')
            ax5.grid(True, alpha=0.3)
            
            # 6. MAPE Distribution for Raw Fed Models
            ax6 = axes[1, 2]
            results_df.boxplot(column='mape', by='model', ax=ax6)
            ax6.set_xlabel('Model Type')
            ax6.set_ylabel('MAPE (%)')
            ax6.set_title('Raw Data Error Distribution')
            ax6.grid(True, alpha=0.3)
            plt.setp(ax6.xaxis.get_majorticklabels(), rotation=45, ha='right')
            
            plt.tight_layout()
            plt.savefig(r'C:\Users\elect\OneDrive\Documentos\Doctorado\Articulo 2 peer review\Images\baseline_raw_fed_inflation_analysis.png', dpi=300, bbox_inches='tight')
            plt.show()
            
            # ================================================
            # 11. BASELINE SCIENTIFIC REPORT
            # ================================================
            
            print(f"\n📋 BASELINE FED SCIENTIFIC ANALYSIS REPORT")
            print("="*75)
            
            print(f"\n🎯 RESEARCH OBJECTIVE:")
            print(f"Establish baseline performance for neural networks on raw Fed indicators")
            print(f"without preprocessing filters (comparison reference for filtered approaches)")
            
            print(f"\n📊 EXPERIMENTAL SETUP:")
            print(f"   Total experiments: {len(results_df)}")
            print(f"   Models tested: {results_df['model'].nunique()}")
            print(f"   Raw Fed combinations: {results_df['leading_indicators'].nunique()}")
            print(f"   Best MSE achieved: {results_df['mse'].min():.6f}")
            print(f"   Best R² achieved: {results_df['r2'].max():.6f}")
            
            print(f"\n🏆 TOP 5 BASELINE CONFIGURATIONS:")
            top_5 = results_df.nsmallest(5, 'mse')
            for i, (_, row) in enumerate(top_5.iterrows(), 1):
                indicators_str = '+'.join(row['leading_indicators'])
                print(f"{i}. {row['model']} | Raw Fed Indicators: {indicators_str}")
                print(f"   Scaler: {row['scaler']} | MSE: {row['mse']:.6f} | R²: {row['r2']:.4f}")
                print(f"   MAE: {row['mae']:.4f} | MAPE: {row['mape']:.2f}%")
                print("-" * 75)
            
            print(f"\n🤖 BASELINE MODEL RANKING:")
            model_ranking = results_df.groupby('model').agg({
                'mse': ['mean', 'std'],
                'r2': ['mean', 'std'],
                'mape': ['mean', 'std']
            }).round(6)
            print(model_ranking)
            
            print(f"\n📈 BASELINE KEY FINDINGS:")
            best_config = results_df.loc[results_df['mse'].idxmin()]
            print(f"   • Best baseline model: {best_config['model']}")
            print(f"   • Optimal raw Fed indicators: {'+'.join(best_config['leading_indicators'])}")
            print(f"   • Best normalization: {best_config['scaler']}")
            
            print(f"\n💡 BASELINE IMPLICATIONS:")
            print(f"   • Establishes performance floor for filtered approaches")
            print(f"   • Shows neural network capability on raw noisy data")
            print(f"   • Provides reference for filter effectiveness evaluation")
            print(f"   • Demonstrates importance of regularization for raw data")
            
            print(f"\n🔬 SCIENTIFIC CONTRIBUTION:")
            print(f"   • Rigorous baseline for filter comparison studies")
            print(f"   • Raw data performance benchmarks for Fed indicators")
            print(f"   • Methodology for establishing unfiltered baselines")
            print(f"   • Framework for evaluating preprocessing effectiveness")

    except Exception as e:
        print(f"❌ Error during baseline analysis: {e}")
        import traceback
        traceback.print_exc()

print(f"\n🎯 BASELINE FED INFLATION FORECASTING COMPLETED!")
print("📊 Raw data baseline established for comprehensive filter comparison")
print("🔬 Results provide reference point for HP, Kalman, and Baxter-King evaluation")
print("🏦 Baseline methodology applicable to all Fed forecasting studies")

🚀 BASELINE INFLATION FORECASTING WITHOUT FILTERS
✅ NO inflation as predictor (avoiding data leakage)
✅ Raw Fed indicators without preprocessing filters
✅ Multiple evaluation metrics (MSE, MAE, R², MAPE)
✅ Baseline for comparison with filtered approaches
✅ Direct neural network learning from unfiltered data

🎯 BASELINE RESEARCH DESIGN (NO FILTERS):
Leading Fed Indicators (Raw): ['USM2', 'USPPIYY', 'INDPRO', 'UNRATE', 'USOIL']
Target Variable: Annual Inflation Rate

💡 Research Question:
How do neural networks perform on raw Fed indicators without preprocessing?
This provides the baseline for comparison with filtered approaches.

🤖 MODEL ARCHITECTURES FOR RAW FED DATA:
   LSTM: Sequential: LSTM(50) → LSTM(50) → Dense(1)
     └─ Optimal for: Long-term dependencies in raw economic data
   GRU: Sequential: GRU(50) → GRU(50) → Dense(1)
     └─ Optimal for: Efficient processing of raw temporal patterns
   CNN: Conv1D(64) → Flatten → Dense(50) → Dense(1)
     └─ Optimal for: Local pattern detec